# Skeletonized Morphology Visualization Notebook

Copyright (c) 2025 Open Brain Institute

+ Author(s): 
    - Michael W. Reimann < michael.reimann@openbraininstitute.org >
    - Marwan Abdellah < marwan.abdellah@openbraininstitute.org >

Last modified: 03.2026

## Imports and setting up platform authentication

We begin by importing required packages.

In [ ]:
import os
import obi_auth
from obi_notebook import get_projects, get_entities, get_environment
from obi_one.scientific.from_id.cell_morphology_from_id import CellMorphologyFromID

from entitysdk import Client
from entitysdk.models import CellMorphology, Subject, EMCellMesh, SkeletonizationExecution
from entitysdk.types import ContentType

from morph_spines import load_morphology_with_spines
import morph_spines_visualizer
from ipywidgets import widgets

import pandas as pd
from IPython.display import display

import logging
loggers = [logging.getLogger(name) for name in logging.root.manager.loggerDict]
for logger in loggers:
    logger.setLevel(logging.ERROR)

## Authentication and project selection
We authenticate with the OBI platform.
### Project selection
Select from the dropdown menu the project that the output (a morphology with extracted spines) should be registered to. It will be available only in that project context. 

The widget lists all projects you have access to.

In [ ]:
environment = get_environment.get_environment()
token = obi_auth.get_token(environment=environment, auth_mode="daf")
project_context = get_projects.get_projects(token, env=environment)

## Set up clients

With the information provided above we assemble a client that can interact with the entity database.

In [ ]:
client = Client(environment=environment, token_manager=token, project_context=project_context)

## Morphology selection

Select the morphology for visualization.

In [ ]:
m = [(m_.name, CellMorphologyFromID(id_str=str(m_.id))) for m_ in 
     client.search_entity(entity_type=CellMorphology, query={
        "has_segmented_spines": True
     })]
m = [m_ for m_ in m if m_[1].has_source_mesh(client)]

if len(m) == 0:
    display(widgets.HTML("""
    <p style="background-color:Tomato;">Unfortunately, you have no access to any morphology with segmented spines.<br>
    Try selecting another project or running the "Skeletonization" workflow!</p>"""))
else:
    select_morphology = widgets.Dropdown(options=m)
    display(widgets.HTML("""
    <p style="background-color:Green;">Select a morphology with segmented spines from the list below.<br>
    To generate more morphologies with spines, please run the "Skeletonization" workflow!</p>"""))
    display(select_morphology)

## Morphology download

Download the selected morphology and its mesh

In [ ]:
root = os.path.join(os.environ["HOME"], "skeletonization_download")
os.makedirs(root, exist_ok=True)

morphology = select_morphology.value
spiny_morph_name = os.path.join(root, morphology.entity(client).name + ".h5")
morphology.write_spiny_neuron_h5(spiny_morph_name, client)

## Find and download source mesh

In [ ]:
source_mesh = morphology.source_mesh_entity(client)

assets = {asset.content_type: asset for asset in source_mesh.assets}
if ContentType.application_obj in assets:
    mesh_file = client.download_file(entity_id=source_mesh.id, entity_type=EMCellMesh,
                                     asset_id=assets[ContentType.application_obj],
                                     output_path=root)

## Visualize the spiny morphology 
This visualization plots the skeleton of the resulting morphology combined with the EM mesh. Users can then select any section with spines, and the spine meshes will pop-up in the scene. 

In [ ]:
# Visualize the data 
morph_spines_visualizer.visualize_morphology_with_point_cloud(
    morphology_path=spiny_morph_name, 
    mesh_path=mesh_file
)